In [2]:
import numpy as np
import xarray as xr
import dask.array as da

In [3]:
# Create the data array
# data = xr.DataArray(
#     np.random.rand(5, 6),
#     dims=["time", "variable"],
#     coords={
#         "time": np.arange(5),
#         "variable": np.arange(6),

#     },
# )

data = xr.open_zarr("/mnt/home/sd5313/data/Ocean/3D_data")

In [4]:
data

<xarray.Dataset>
Dimensions:        (time: 4745, y: 180, x: 360)
Coordinates:
  * time           (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x              (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y              (y) float64 -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
Data variables: (12/80)
    hfds           (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_0       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_1       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_10      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_11      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_12      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    ...             ...
    vo_lev_5       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_6       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_7       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_8       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_9       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    zos            (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>

In [23]:
%%timeit
# Define slices
slices = [slice(0, 2), slice(1, 3), slice(2, 4)]  # Same length and continuous slices

# Select the data for each slice
sliced_data = [
    data.isel(time=slc).assign_coords(time=np.arange(slc.stop - slc.start))
    for slc in slices
]

# Concatenate along the new dimension 'window_dim'
result = xr.concat(sliced_data, dim="window_dim")

result.load()

1.04 s ± 43.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [24]:
%%timeit
def slice_indices(size, slices):
    range_ = range(size)
    return np.array([range_[slice_] for slice_ in slices])


slices = [slice(0, 2), slice(1, 3), slice(2, 4)]
dims = ["window_dim", "time"]
d = data.isel(time=xr.Variable(dims, slice_indices(data.sizes["time"], slices)))
d.load()

746 ms ± 21.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [23]:
%%time

# Define slices
window_size = 2
step = 1
slices = np.arange(0, data.sizes["time"] - window_size + 1, step)

data = data.assign_coords(time=np.arange(data.sizes["time"]))

# Use advanced indexing to select overlapping slices
sliced_data = data.isel(
    time=xr.DataArray(
        slices[:, None] + np.arange(window_size), dims=["window_dim", "window_step"]
    )
)

# Adjust the coordinates
result2 = sliced_data.assign_coords(time=sliced_data.time - slices[:, None])
# print(result2)

CPU times: user 5.67 s, sys: 13.1 ms, total: 5.69 s
Wall time: 5.69 s


In [11]:
%%time
# Define slices
window_size = 2
step = 1
slices = np.arange(0, data.sizes["time"] - window_size + 1, step)

# Use advanced indexing to select overlapping slices
sliced_data = data.isel(
    time=xr.DataArray(
        slices[:, None] + np.arange(window_size), dims=["window_dim", "window_step"]
    )
)

# Adjust the coordinates
# result = sliced_data.assign_coords(time=sliced_data.time - slices[:, None])
data = data.assign_coords(time=np.arange(data.sizes["time"]))

# Add Window dim as a coordinate
result = sliced_data.assign_coords(
    window_dim=np.arange(sliced_data.sizes["window_dim"])
)

# Trigger computation
result.isel(window_dim=0).load()
# print(result)

KeyboardInterrupt: 

In [22]:
%%timeit
total_steps = 1 + 1
rolling_dataset = data.rolling(
    time=len(data.time) - total_steps, center=False
).construct("window_dim")
rolling_dataset = rolling_dataset.isel(
    time=slice(len(data.time) - total_steps - 1, None)
)
data_in = rolling_dataset.isel(window_dim=slice(None, 2), time=slice(None, -1))
# data_in = ((data_in - self.in_no_extra_mean) / self.in_no_extra_std).fillna(0)
data_in = data_in.transpose("window_dim", "time", "variable").to_numpy()

885 µs ± 11 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
